# Analyse movement bouts
The goal here is to check the shape of the data, specifically if we're able to run the analysis that we want.
## Aim
En substance, we would like to:
* Extract pre-motor activity in moments that the animal is otherwise (relatively) immobile, both in tracking and playback
* Guess features of movement from this activity, such as direction, speed, distance to final position, etc.
* This could be done via dimensionality reduction, direct regression, other sophisticated tech.
## Limits
The list of answerable questions is large, but we need to start by figuring out if the data is plentyful enough

## Parameters
We will select:

**Immobility**

Absolute position velocity stays continuously below 0.02 position units/s for at least 1.2 s.

**Movement onset**

The first sample following a qualifying immobility period. We apply the same definition in tracking and playback.

**Movement endpoint and final position**

Provisionally, the endpoint is the start of the next qualifying immobility period, and final position is Position at that sample. If none follows before the session ends, the endpoint stays missing and the onset remains available for pre-motion analysis.

## Saved info:
We record:
* Identification data: animal, headstage, source session
* Condition: i.e., playback or tracking (0/1, as is already in feature). Set at movement onset (to avoid complicated mixed case)
* onset bin/time
* end bin/time
* Initial position
* Final position
* Total movement
* Direction (initial, to avoid def. problems if it changes, so what matters is the first steps. might be redefined later)
* Speed (avg speed, I guess?)


# 1. Load data

In [ ]:
from pathlib import Path
import pickle

import pandas as pd
from IPython.display import display

# Locate the shared data folder from the workspace, repo, or notebooks folder.
data_dir = next(
    (parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data").is_dir()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Cannot locate the workspace data directory.")

feature_paths = sorted(data_dir.glob("*_AC_playback_0.005_feature_ss"))
if not feature_paths:
    raise FileNotFoundError(f"No AC feature files found in {data_dir}")

dt = 0.005  # seconds per bin
features = []
session_rows = []
for path in feature_paths:
    animal, _, headstage, region, *_ = path.name.split("_")
    with path.open("rb") as stream:
        collection_features = pickle.load(stream)

    for session, session_features in enumerate(collection_features):
        features.append(session_features)
        session_rows.append({
            "animal": animal,
            "headstage": int(headstage),
            "region": region,
            "session": session,
            "feature_file": path.name,
            "n_bins": len(session_features),
            "duration_s": len(session_features) * dt,
            "tracking_s": session_features["Condition"].eq(0).sum() * dt,
            "playback_s": session_features["Condition"].eq(1).sum() * dt,
        })
    del collection_features, session_features

# features[s][column] corresponds to session_info.loc[s].
# Session entries count each headstage separately.
session_info = pd.DataFrame(session_rows)
print(f"Loaded {len(features)} AC session entries from {len(feature_paths)} collections.")
display(session_info)


# 2. Do itttt

In [ ]:
# Enter parameters for analysis
DT = 0.005
# Speed_x uses position differences * 100. For 5 ms bins, * 2 converts to position units/s.
SPEED_X_FACTOR = 2.0

VELOCITY_THRESHOLD = 0.02
MIN_IMMOBILITY_S = 1.2

In [ ]:
# let's work here while we design a helper function. For now we'll just make the code work on one specific session

idx = 0
# we access features of a specific session by direct ref indexing. So for example: features[idx]["Position"]

# First step: define a vector of immobility as booleans. Every bin will be defined as immobile or not from its instantaneous speed.


import numpy as np
import matplotlib.pyplot as plt

idx = 0

position = features[idx]["Position"].to_numpy()
time = np.arange(len(position)) * DT
speed = features[idx]["Speed_x"].to_numpy() * SPEED_X_FACTOR

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(time, position)
axes[0].set_ylabel("Position")

axes[1].plot(time, speed)
axes[1].set_ylabel("Speed (position units/s)")
axes[1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()



In [ ]:
idx = 0
DT = 0.005

position = features[idx]["Position"].to_numpy()
stored_speed = features[idx]["Speed_x"].to_numpy()

velocity = np.diff(position, prepend=np.nan) / DT
derived_speed = np.abs(velocity)
time = np.arange(len(position)) * DT

plt.figure(figsize=(14, 4))
plt.plot(time, derived_speed, label="Speed from position", alpha=0.7)
plt.plot(time, stored_speed, label="Stored Speed_x", alpha=0.7)
plt.xlabel("Time (s)")
plt.ylabel("Speed")
plt.legend()
plt.show()

moving = (
    np.isfinite(derived_speed)
    & np.isfinite(stored_speed)
    & (stored_speed > 0)
)
if moving.any():
    print(
        "Median derived / stored ratio:",
        np.median(derived_speed[moving] / stored_speed[moving]),
    )